# O3 ablation: raw vs background-subtracted CSI — hrc_ur10_cell

Trains the **same** multi-task model (Stage-1 backbone + 3 heads) twice,
from the same initialization, on the two inputs, and compares them:
- **bgsub** — empty room subtracted (isolates the people).
- **raw** — the full CSI (keeps the static room, i.e. absolute-position cues).

**Hypothesis:** bgsub wins presence/count (isolates the target), but
**raw may help ZONE**, because knowing *where* a person is relative to
the robot needs the absolute-geometry information that bgsub removes.

> **Run in a FRESH Colab runtime** (Runtime -> Change runtime type ->
> GPU), SEPARATE from the Sionna data-generation notebook (its
> numpy==2.0.2 pin breaks stock torch). No Sionna here. Upload this
> notebook + `hrc_ur10_cell_data_v0_robot.npz` into /content.

In [ ]:
import os, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix, recall_score

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

DATA_NAME = 'hrc_ur10_cell_data_v0_robot.npz'
if _in_colab():
    hits = list(Path('/content').rglob(DATA_NAME))
    if not hits: raise FileNotFoundError(f'{DATA_NAME} not found under /content.')
    NPZ_PATH = str(hits[0])
else:
    NPZ_PATH = os.path.join(os.getcwd(), DATA_NAME)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEVICE, '| data', NPZ_PATH)

In [ ]:
# ---- config (same as csi_multitask_v0) ----
INPUTS = ['bgsub', 'raw']
BATCH, EPOCHS, LR, WEIGHT_DECAY = 128, 100, 3e-4, 1e-4
DROPOUT_P, SUBCARRIER_DROPOUT = 0.2, 0.1
STEM_CH, STAGE_CH, STAGE_DS = 32, [32, 64, 96, 96], [False, True, True, True]
N_COUNT, N_ZONE = 3, 4
ZONE_NAMES = ['none', 'green', 'yellow', 'red']

# ---- shared labels/splits (same for both inputs) ----
z = np.load(NPZ_PATH, allow_pickle=True)
presence = z['presence'].astype(np.int64); count = z['count'].astype(np.int64)
zone = z['zone'].astype(np.int64); split = z['split'].astype(str)
N, M_AP, K_TX, N_SUB = z['csi_bgsub'].shape
tr, va, te = split == 'train', split == 'val', split == 'test'
print('N', N, '| train/val/test', int(tr.sum()), int(va.sum()), int(te.sum()))

In [ ]:
# ---- model (Stage-1 backbone + shared neck + 3 heads) ----
class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride_w=1):
        super().__init__(); s = (1, stride_w)
        self.conv1 = nn.Conv2d(cin, cout, 3, stride=s, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(cout)
        self.proj = None
        if cin != cout or stride_w != 1:
            self.proj = nn.Sequential(nn.Conv2d(cin, cout, 1, stride=s, bias=False), nn.BatchNorm2d(cout))
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        idt = x if self.proj is None else self.proj(x)
        o = self.act(self.bn1(self.conv1(x))); o = self.bn2(self.conv2(o)); return self.act(o + idt)

class CSINetMulti(nn.Module):
    def __init__(self, in_ch=8, p=DROPOUT_P):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(in_ch, STEM_CH, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(STEM_CH), nn.ReLU(inplace=True))
        blocks, c = [], STEM_CH
        for cout, ds in zip(STAGE_CH, STAGE_DS):
            blocks.append(BasicBlock(c, cout, 2 if ds else 1)); c = cout
        self.stages = nn.Sequential(*blocks); self.pool = nn.AdaptiveAvgPool2d(1)
        self.neck = nn.Sequential(nn.Linear(c, 64), nn.ReLU(inplace=True), nn.Dropout(p))
        self.hp = nn.Linear(64, 1); self.hc = nn.Linear(64, N_COUNT); self.hz = nn.Linear(64, N_ZONE)
    def forward(self, x):
        f = self.neck(self.pool(self.stages(self.stem(x))).flatten(1))
        return self.hp(f).squeeze(1), self.hc(f), self.hz(f)

In [ ]:
# ---- one experiment: prep input -> train -> test ----
def to_input(c):
    s = np.stack([c.real, c.imag], axis=1); s = np.transpose(s, (0, 1, 3, 2, 4))
    return s.reshape(c.shape[0], 2 * K_TX, M_AP, N_SUB).astype(np.float32)

def loaders_for(Xn):
    def mk(mask, sh):
        ds = TensorDataset(torch.tensor(Xn[mask]), torch.tensor(presence[mask]),
                           torch.tensor(count[mask]), torch.tensor(zone[mask]))
        return DataLoader(ds, batch_size=BATCH, shuffle=sh)
    return mk(tr, True), mk(va, False), mk(te, False)

@torch.no_grad()
def evaluate(model, dl):
    model.eval(); P = {k: [] for k in 'pcz'}; T = {k: [] for k in 'pcz'}
    for xb, yp, yc, yz in dl:
        pl, cl, zl = model(xb.to(DEVICE))
        P['p'].append((torch.sigmoid(pl) > 0.5).long().cpu()); T['p'].append(yp)
        P['c'].append(cl.argmax(1).cpu()); T['c'].append(yc)
        P['z'].append(zl.argmax(1).cpu()); T['z'].append(yz)
    P = {k: torch.cat(v).numpy() for k, v in P.items()}; T = {k: torch.cat(v).numpy() for k, v in T.items()}
    return P, T

def run_experiment(INPUT):
    csi = z['csi_bgsub'] if INPUT == 'bgsub' else z['csi_raw']
    X = to_input(csi)
    xm = X[tr].mean((0, 2, 3), keepdims=True); xs = X[tr].std((0, 2, 3), keepdims=True) + 1e-6
    Xn = (X - xm) / xs
    dl_tr, dl_va, dl_te = loaders_for(Xn)
    zc = np.bincount(zone[tr], minlength=N_ZONE).astype(np.float32)
    zw = zc.sum() / (N_ZONE * np.maximum(zc, 1)); zw = torch.tensor(zw / zw.mean(), dtype=torch.float32, device=DEVICE)

    torch.manual_seed(0); np.random.seed(0)      # same init for both inputs (fair ablation)
    model = CSINetMulti().to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    bce, ce, ce_z = nn.BCEWithLogitsLoss(), nn.CrossEntropyLoss(), nn.CrossEntropyLoss(weight=zw)
    def sc_do(xb):
        if SUBCARRIER_DROPOUT <= 0: return xb
        return xb * (torch.rand(xb.shape[-1], device=xb.device) > SUBCARRIER_DROPOUT).float()

    best, best_state = -1, None
    for ep in range(EPOCHS):
        model.train()
        for xb, yp, yc, yz in dl_tr:
            xb = sc_do(xb.to(DEVICE)); yp, yc, yz = yp.float().to(DEVICE), yc.to(DEVICE), yz.to(DEVICE)
            pl, cl, zl = model(xb)
            loss = bce(pl, yp) + ce(cl, yc) + ce_z(zl, yz)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        P, T = evaluate(model, dl_va)
        meanb = np.mean([balanced_accuracy_score(T[k], P[k]) for k in 'pcz'])
        if meanb > best:
            best = meanb; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    P, T = evaluate(model, dl_te)
    res = {'input': INPUT,
           'presence_bacc': balanced_accuracy_score(T['p'], P['p']),
           'count_bacc': balanced_accuracy_score(T['c'], P['c']),
           'zone_bacc': balanced_accuracy_score(T['z'], P['z']),
           'zone_f1': f1_score(T['z'], P['z'], average='macro'),
           'pres_recall': recall_score(T['p'], P['p'], pos_label=1),
           'red_recall': recall_score(T['z'] == 3, P['z'] == 3),
           'zone_cm': confusion_matrix(T['z'], P['z'])}
    return res

In [ ]:
# ---- run both inputs ----
results = {}
for INPUT in INPUTS:
    t0 = time.time()
    results[INPUT] = run_experiment(INPUT)
    print(f"{INPUT:6s} done in {(time.time()-t0)/60:.1f} min | "
          f"zone bal-acc {results[INPUT]['zone_bacc']:.3f} | red-recall {results[INPUT]['red_recall']:.3f}")

In [ ]:
# ---- comparison table + zone confusion side-by-side ----
cols = ['presence_bacc', 'count_bacc', 'zone_bacc', 'zone_f1', 'pres_recall', 'red_recall']
print(f"{'input':6s} " + '  '.join(f'{c:>12s}' for c in cols))
for INPUT in INPUTS:
    r = results[INPUT]
    print(f"{INPUT:6s} " + '  '.join(f'{r[c]:12.3f}' for c in cols))

d_zone = results['raw']['zone_bacc'] - results['bgsub']['zone_bacc']
d_red = results['raw']['red_recall'] - results['bgsub']['red_recall']
print(f"\nraw - bgsub:  zone bal-acc {d_zone:+.3f} | red-recall {d_red:+.3f}")
print('=> hypothesis supported if raw improves zone / red-recall.')

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, INPUT in zip(ax, INPUTS):
    cm = results[INPUT]['zone_cm']; im = a.imshow(cm, cmap='Blues')
    a.set_title(f'zone confusion — {INPUT}'); a.set_xlabel('predicted'); a.set_ylabel('true')
    a.set_xticks(range(4)); a.set_xticklabels(ZONE_NAMES, rotation=45)
    a.set_yticks(range(4)); a.set_yticklabels(ZONE_NAMES)
    for (rr, cc), v in np.ndenumerate(cm):
        a.text(cc, rr, int(v), ha='center', va='center', color='white' if v > cm.max()/2 else 'black')
fig.tight_layout(); plt.show()

## What to read

- **`raw - bgsub` on zone bal-acc / red-recall** is the headline. If
  positive, the absolute-position cues in raw CSI genuinely help the
  spatial (zone) task, and we should feed raw (or raw+bgsub) to the
  zone head.
- **Presence/count** are expected to stay high for both; bgsub may keep
  a small edge there.
- Look at the **green↔red off-diagonal** in the zone confusions — the
  safety-critical error. Fewer red→green cells is better.

If raw helps zone, the natural next model is a **two-input** design
(bgsub for presence/count, raw for zone) — but only if the ablation
justifies the extra complexity.